<a href="https://colab.research.google.com/github/olayemiojo/skills-copilot-codespaces-vscode/blob/main/2%2C_3%2C_4_Internal_Knowledge_Base_Agents_with_OpenAI_%26_LangChain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Internal Knowledge Base Agents with OpenAI & LangChain

## Dependency Installs

In [ ]:
%pip install langchain
%pip install chromadb
%pip install pypdf
%pip install colab-env
%pip install openai
%pip install tiktoken
%pip install pydantic

## Files setup

- In the left side menu of the Colab notebook click the Files icon
- Create a new folder by right-clicking in the blank area and create a folder titled `PDFs`
- Place any PDFs you'd like in the folder by using the file upload button (Document with arrow pointing up)
- These PDFs will be used for creating the vector store and you can query knowledge from them re-writing the statements in the code blocks below where appropriate

## API Key Setup

- Create a file on your computer titled `vars.env`
- Obtain your OpenAI API Key from this link, https://platform.openai.com/account/api-keys and copy your API key you generate
- Place the following as the contents of your file and paste the value of your key after the `=` where `YOUR-API-KEY` is
```
OPENAI_API_KEY=YOUR-API-KEY
```
- Upload the `vars.env` file to the root folder of your Google Drive and the rest takes care of itself!

# Loading the PDFs and Splitting

In [ ]:
from langchain.document_loaders import PyPDFLoader
from langchain.document_loaders import DirectoryLoader
from langchain.text_splitter import CharacterTextSplitter

loader = DirectoryLoader('PDFs/', glob="*.pdf", loader_cls=PyPDFLoader)
docs = loader.load_and_split()

page_contents = [doc.page_content for doc in docs]
metadatas = [doc.metadata for doc in docs]

# Initialize the text splitter
text_splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=150,
    chunk_overlap=20,
    length_function=len,
)

# Call the create_documents method with the extracted lists
split_documents = text_splitter.create_documents(page_contents, metadatas=metadatas)

# Creating the vector store

In [ ]:
import colab_env
import os
import openai
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.vectorstores import Chroma

openai.api_key = os.getenv('OPENAI_API_KEY')

embeddings = OpenAIEmbeddings()

split_metadatas = [{"source": f"Document: {doc.metadata['source']}, Page: {doc.metadata['page'] + 1}"} for doc in split_documents]
split_documents_texts = [doc.page_content for doc in split_documents]

docsearch = Chroma.from_texts(texts=split_documents_texts, embedding=embeddings, metadatas=split_metadatas)

query = "How much space should be between the table and backdrop?"
docs = docsearch.similarity_search(query)

# Creating the Retrieval QA Chain

In [ ]:
from langchain.chains import RetrievalQAWithSourcesChain
from langchain import OpenAI

chain = RetrievalQAWithSourcesChain.from_chain_type(OpenAI(model='text-davinci-003', temperature=0), chain_type="stuff", retriever=docsearch.as_retriever())

chain({"question": "How much space should be between the table and backdrop?"}, return_only_outputs=True)

                    model was transfered to model_kwargs.
                    Please confirm that model is what you intended.


{'answer': ' 3-4 feet between the table and backdrop.\n',
 'sources': 'Document: PDFs/Remote Studio Setup Guide.pdf, Page: 16'}

# Creating a Tool from our Retrieval Chain

In [ ]:
from langchain.tools import BaseTool

class KnowledgeBaseTool(BaseTool):
    name = "Knowledge Base Tool"
    description = "useful for answering questions from about stormwind studios"

    def _run(self, query: str) -> str:
        """Use the tool."""
        return chain({"question": query}, return_only_outputs=True)

    async def _arun(self, query: str) -> str:
        """Use the tool asynchronously."""
        raise NotImplementedError("Knowledge Base Tool does not support async")

tools = [KnowledgeBaseTool()]

# Creating our Agent Prompt

In [ ]:
# Set up the base template
template = """Answer the following questions as best you can as an agent of Stormwind Studios
helping its contractors and employees with their questions:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin! Remember if you see an observation that contains content and a source be helpful
and tell them what document has the answer and what page the answer is on

Question: {input}
{agent_scratchpad}"""

In [ ]:
from langchain.prompts import StringPromptTemplate
from typing import List

# Set up a prompt template
class CustomPromptTemplate(StringPromptTemplate):
    # The template to use
    template: str
    # The list of tools available
    tools: List[BaseTool]

    def format(self, **kwargs) -> str:
        # Get the intermediate steps (AgentAction, Observation tuples)
        # Format them in a particular way
        intermediate_steps = kwargs.pop("intermediate_steps")
        thoughts = ""
        for action, observation in intermediate_steps:
            thoughts += action.log
            thoughts += f"\nObservation: {observation}\nThought: "
        # Set the agent_scratchpad variable to that value
        kwargs["agent_scratchpad"] = thoughts
        # Create a tools variable from the list of tools provided
        kwargs["tools"] = "\n".join([f"{tool.name}: {tool.description}" for tool in self.tools])
        # Create a list of tool names for the tools provided
        kwargs["tool_names"] = ", ".join([tool.name for tool in self.tools])
        return self.template.format(**kwargs)

In [ ]:
prompt = CustomPromptTemplate(
    template=template,
    tools=tools,
    input_variables=["input", "intermediate_steps"]
)

# Creating the output parser

In [ ]:
from typing import List, Union
from langchain.agents import AgentOutputParser
from langchain.schema import AgentAction, AgentFinish
import re

class CustomOutputParser(AgentOutputParser):

    def parse(self, llm_output: str) -> Union[AgentAction, AgentFinish]:
        # Check if agent should finish
        if "Final Answer:" in llm_output:
            return AgentFinish(
                # Return values is generally always a dictionary with a single `output` key
                # It is not recommended to try anything else at the moment :)
                return_values={"output": llm_output.split("Final Answer:")[-1].strip()},
                log=llm_output,
            )
        # Parse out the action and action input
        regex = r"Action\s*\d*\s*:(.*?)\nAction\s*\d*\s*Input\s*\d*\s*:[\s]*(.*)"
        match = re.search(regex, llm_output, re.DOTALL)
        if not match:
            raise ValueError(f"Could not parse LLM output: `{llm_output}`")
        action = match.group(1).strip()
        action_input = match.group(2)
        # Return the action and action input
        return AgentAction(tool=action, tool_input=action_input.strip(" ").strip('"'), log=llm_output)

output_parser = CustomOutputParser()

# Instantiate the agent

In [ ]:
from langchain.agents import LLMSingleActionAgent, AgentExecutor
from langchain import LLMChain
from langchain.chat_models import ChatOpenAI

llm = ChatOpenAI(model='gpt-4', temperature=0)
# LLM chain consisting of the LLM and a prompt
llm_chain = LLMChain(llm=llm, prompt=prompt)

tool_names = [tool.name for tool in tools]
agent = LLMSingleActionAgent(
    llm_chain=llm_chain,
    output_parser=output_parser,
    stop=["\nObservation:"],
    allowed_tools=tool_names
)

In [ ]:
agent_executor = AgentExecutor.from_agent_and_tools(agent=agent, tools=tools, verbose=True)
agent_executor.run("How much space should be between the table and backdrop?")



> Entering new AgentExecutor chain...
Thought: I need to find the information about the space between the table and backdrop.
Action: Knowledge Base Tool
Action Input: space between table and backdrop

Observation:{'answer': ' The recommended space between the table and backdrop is 3-4 feet.\n', 'sources': 'Document: PDFs/Remote Studio Setup Guide.pdf, Page: 16'}
I now know the final answer.
Final Answer: The recommended space between the table and backdrop is 3-4 feet. You can find this information in the Remote Studio Setup Guide on page 16.

> Finished chain.


'The recommended space between the table and backdrop is 3-4 feet. You can find this information in the Remote Studio Setup Guide on page 16.'

# Adding memory to the Agent

In [ ]:
from langchain.memory import ConversationBufferWindowMemory

# Set up the base template WITH memory
template_with_history = """Answer the following questions as best you can as an agent of Stormwind Studios
helping its contractors and employees with their questions:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin! Remember if you see an observation that contains content and a source be helpful
and tell them what document has the answer and what page the answer is on

Previous conversation history:
{history}

Question: {input}
{agent_scratchpad}"""

prompt_with_history = CustomPromptTemplate(
    template=template_with_history,
    tools=tools,
    # This omits the `agent_scratchpad`, `tools`, and `tool_names` variables because those are generated dynamically
    # This includes the `intermediate_steps` variable because that is needed
    input_variables=["input", "intermediate_steps", "history"]
)

llm_chain = LLMChain(llm=llm, prompt=prompt_with_history)

tool_names = [tool.name for tool in tools]
agent = LLMSingleActionAgent(
    llm_chain=llm_chain,
    output_parser=output_parser,
    stop=["\nObservation:"],
    allowed_tools=tool_names
)

memory=ConversationBufferWindowMemory(k=4)

agent_executor = AgentExecutor.from_agent_and_tools(agent=agent, tools=tools, verbose=True, memory=memory)

In [ ]:
agent_executor.run("What's inside the lighting kit?")



> Entering new AgentExecutor chain...
Thought: I need to find information about the contents of the lighting kit.
Action: Knowledge Base Tool
Action Input: What's inside the lighting kit?

Observation:{'answer': ' The lighting kit contains 2 bulbs in white boxes, 2 stands, 2 “Softbox” light heads, and 2 diffusion screens folded in plastic bags.\n', 'sources': 'Document: PDFs/Remote Studio Setup Guide.pdf, Pages: 3, 4, 5'}
I now know the contents of the lighting kit.
Final Answer: The lighting kit contains 2 bulbs in white boxes, 2 stands, 2 “Softbox” light heads, and 2 diffusion screens folded in plastic bags. You can find this information in the Remote Studio Setup Guide on pages 3, 4, and 5.

> Finished chain.


'The lighting kit contains 2 bulbs in white boxes, 2 stands, 2 “Softbox” light heads, and 2 diffusion screens folded in plastic bags. You can find this information in the Remote Studio Setup Guide on pages 3, 4, and 5.'

In [ ]:
agent_executor.run("How many slides should I be recording every day when recording?")



> Entering new AgentExecutor chain...
Thought: I need to find information about the number of slides to record daily.
Action: Knowledge Base Tool
Action Input: How many slides should be recorded daily?

Observation:{'answer': ' 40-50 slides should be recorded daily.\n', 'sources': 'Document: PDFs/Contract Instructor Guide.pdf, Page: 2'}
I now know the final answer.
Final Answer: You should aim to record 40-50 slides every day when recording. This information can be found in the Contract Instructor Guide on page 2.

> Finished chain.


'You should aim to record 40-50 slides every day when recording. This information can be found in the Contract Instructor Guide on page 2.'

In [ ]:
agent_executor.run("What kind of file type does the teleprompter work with?")



> Entering new AgentExecutor chain...
Thought: I need to find the file type that the teleprompter works with.
Action: Knowledge Base Tool
Action Input: What file type does the teleprompter work with?

Observation:{'answer': ' The teleprompter works with .rtf files.\n', 'sources': 'Document: PDFs/Contract Instructor Guide.pdf, Page: 2'}
I now know the final answer.
Final Answer: The teleprompter works with .rtf files. You can find this information in the Contract Instructor Guide on page 2.

> Finished chain.


'The teleprompter works with .rtf files. You can find this information in the Contract Instructor Guide on page 2.'